# **Homework 2: Phoneme Classification**


Objectives:
* Solve a classification problem with deep neural networks (DNNs).
* Understand recursive neural networks (RNNs).

If you have any questions, please contact the TAs via TA hours, NTU COOL, or email to mlta-2023-spring@googlegroups.com

# Download Data
Download data from google drive, then unzip it.

You should have
- `libriphone/train_split.txt`: training metadata
- `libriphone/train_labels`: training labels
- `libriphone/test_split.txt`: testing metadata
- `libriphone/feat/train/*.pt`: training feature
- `libriphone/feat/test/*.pt`:  testing feature

after running the following block.

> **Notes: if the google drive link is dead, you can download the data directly from [Kaggle](https://www.kaggle.com/c/ml2023spring-hw2/data) and upload it to the workspace.**


In [ ]:
from pathlib import Path
import zipfile
import gdown

# 寻找仓库根目录
current_dir = Path.cwd()

repo_root = next(
    (
        path
        for path in [current_dir, *current_dir.parents]
        if (path / ".gitignore").exists()
    ),
    current_dir,
)

data_dir = repo_root / "data" / "hw02"
data_dir.mkdir(parents=True, exist_ok=True)

zip_path = data_dir / "libriphone.zip"
dataset_dir = data_dir / "libriphone"

print("Current directory:", current_dir)
print("Repository root:", repo_root)
print("Dataset directory:", dataset_dir)

# 下载数据
if not zip_path.exists():
    result = gdown.download(
        id="1qzCRnywKh30mTbWUEjXuNT2isOCAPdO1",
        output=str(zip_path),
        quiet=False,
    )

    if result is None:
        raise RuntimeError("libriphone.zip 下载失败")
else:
    print("压缩包已经存在：", zip_path)

# 解压数据
if not dataset_dir.exists():
    print("正在解压……")

    with zipfile.ZipFile(zip_path, "r") as zip_file:
        zip_file.extractall(data_dir)

    print("解压完成")
else:
    print("数据集已经解压：", dataset_dir)

Current directory: /home/soyo/HungYi_Lee_ml/homework/hw02_classification
Repository root: /home/soyo/HungYi_Lee_ml
Dataset directory: /home/soyo/HungYi_Lee_ml/data/hw02/libriphone


# Some Utility Functions
**Fixes random number generator seeds for reproducibility.**

In [2]:
import numpy as np
import torch
import random

def same_seeds(seed):
    random.seed(seed) 
    np.random.seed(seed)  
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed) 
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

**Helper functions to pre-process the training data from raw MFCC features of each utterance.**

A phoneme may span several frames and is dependent to past and future frames. \
Hence we concatenate neighboring phonemes for training to achieve higher accuracy. The **concat_feat** function concatenates past and future k frames (total 2k+1 = n frames), and we predict the center frame.

Feel free to modify the data preprocess functions, but **do not drop any frame** (if you modify the functions, remember to check that the number of frames are the same as mentioned in the slides)

In [ ]:
import os
import torch
from tqdm import tqdm

def load_feat(path):
    feat = torch.load(path)
    return feat
# shif函数负责将输入张量x沿着时间维度进行平移，平移的步数由参数n指定。具体来说：
# 如果n为负数，则将x向左平移-n个时间步，并在右侧填充x的第一个时间步的值。
# 如果n为正数，则将x向右平移n个时间步，并在左侧填充x的最后一个时间步的值。
# 如果n为零，则不进行平移，直接返回原始张量x。
def shift(x, n):
    if n < 0:
        left = x[0].repeat(-n, 1)
        right = x[:n]
    elif n > 0:
        right = x[-1].repeat(n, 1)
        left = x[n:]
    else:
        return x

    return torch.cat((left, right), dim=0)
# concat_feat函数用于将输入张量x沿着时间维度进行特征拼接。具体来说：
# 1. 首先检查concat_n是否为奇数，如果不是则抛出异常。
# 2. 如果concat_n小于2，则直接返回原始张量x。
# 3. 将输入张量x沿着特征维度重复concat_n次。
# 4. 将重复后的张量重新排列为一个三维张量，形状为(concat_n, seq_len, feature_dim)，其中seq_len是时间步的长度，feature_dim是特征维度的大小。
# 5. 对于每个时间步，使用shift函数将其向左和向右平移相应的步数，并将平移后的结果存储在新的张量中。
# 6. 最后，将拼接后的张量重新排列为一个二维张量，形状为(seq_len, concat_n * feature_dim)，并返回该张量。
def concat_feat(x, concat_n):
    assert concat_n % 2 == 1 # n must be odd
    if concat_n < 2:
        return x
    seq_len, feature_dim = x.size(0), x.size(1)
    x = x.repeat(1, concat_n) 
    x = x.view(seq_len, concat_n, feature_dim).permute(1, 0, 2) # concat_n, seq_len, feature_dim
    mid = (concat_n // 2)
    for r_idx in range(1, mid+1):
        x[mid + r_idx, :] = shift(x[mid + r_idx], r_idx)
        x[mid - r_idx, :] = shift(x[mid - r_idx], -r_idx)

    return x.permute(1, 0, 2).view(seq_len, concat_n * feature_dim)
# preprocess_data函数用于预处理数据集。它根据指定的拆分（训练、验证或测试）加载特征和标签，并将特征进行拼接处理。具体来说：
# 1. 根据拆分参数确定模式（训练或测试）。
# 2. 如果是训练模式，加载标签字典，并根据训练比例划分训练和验证数据。
# 3. 加载特征文件，并将其拼接为指定的帧数。
# 4. 如果是训练模式，还会加载对应的标签。
# 5. 最后返回处理后的特征和标签（如果是训练模式）
def preprocess_data(split, feat_dir, phone_path, concat_nframes, train_ratio=0.8):
    class_num = 41 # NOTE: pre-computed, should not need change

    if split == 'train' or split == 'val':
        mode = 'train'
    elif split == 'test':
        mode = 'test'
    else:
        raise ValueError('Invalid \'split\' argument for dataset: PhoneDataset!')

    label_dict = {}
    if mode == 'train':
        for line in open(os.path.join(phone_path, f'{mode}_labels.txt')).readlines():
            line = line.strip('\n').split(' ')
            label_dict[line[0]] = [int(p) for p in line[1:]]
        
        # split training and validation data
        usage_list = open(os.path.join(phone_path, 'train_split.txt')).readlines()
        random.shuffle(usage_list)
        train_len = int(len(usage_list) * train_ratio)
        usage_list = usage_list[:train_len] if split == 'train' else usage_list[train_len:]

    elif mode == 'test':
        usage_list = open(os.path.join(phone_path, 'test_split.txt')).readlines()

    usage_list = [line.strip('\n') for line in usage_list]
    print('[Dataset] - # phone classes: ' + str(class_num) + ', number of utterances for ' + split + ': ' + str(len(usage_list)))

    max_len = 3000000
    X = torch.empty(max_len, 39 * concat_nframes)
    if mode == 'train':
        y = torch.empty(max_len, dtype=torch.long)

    idx = 0
    for i, fname in tqdm(enumerate(usage_list)):
        feat = load_feat(os.path.join(feat_dir, mode, f'{fname}.pt'))
        cur_len = len(feat)
        feat = concat_feat(feat, concat_nframes)
        if mode == 'train':
          label = torch.LongTensor(label_dict[fname])

        X[idx: idx + cur_len, :] = feat
        if mode == 'train':
          y[idx: idx + cur_len] = label

        idx += cur_len

    X = X[:idx, :]
    if mode == 'train':
      y = y[:idx]

    print(f'[INFO] {split} set')
    print(X.shape)
    if mode == 'train':
      print(y.shape)
      return X, y
    else:
      return X


In [6]:
x = torch.tensor([
    [1.0],
    [2.0],
    [3.0],
    [4.0],
])

result = concat_feat(x, concat_n=3)

print("原始输入：")
print(x)

print("\n拼接结果：")
print(result)

print("\nshape:", result.shape)

原始输入：
tensor([[1.],
        [2.],
        [3.],
        [4.]])

拼接结果：
tensor([[1., 1., 2.],
        [1., 2., 3.],
        [2., 3., 4.],
        [3., 4., 4.]])

shape: torch.Size([4, 3])


# Dataset

In [7]:
import torch
from torch.utils.data import Dataset

class LibriDataset(Dataset):
    def __init__(self, X, y=None):
        self.data = X
        if y is not None:
            self.label = torch.LongTensor(y)
        else:
            self.label = None

    def __getitem__(self, idx):
        if self.label is not None:
            return self.data[idx], self.label[idx]
        else:
            return self.data[idx]

    def __len__(self):
        return len(self.data)


# Model
Feel free to modify the structure of the model.

In [18]:
import torch.nn as nn

class BasicBlock(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(BasicBlock, self).__init__()

        # TODO: apply batch normalization and dropout for strong baseline.
        # Reference: https://pytorch.org/docs/stable/generated/torch.nn.BatchNorm1d.html (batch normalization)
        #       https://pytorch.org/docs/stable/generated/torch.nn.Dropout.html (dropout)
        self.block = nn.Sequential(
            nn.Linear(input_dim, output_dim),
            nn.BatchNorm1d(output_dim),
            nn.ReLU(),
            nn.Dropout(p=0.5)
        )

    def forward(self, x):
        x = self.block(x)
        return x


class Classifier(nn.Module):
    def __init__(self, input_dim, output_dim=41, hidden_layers=1, hidden_dim=256):
        super(Classifier, self).__init__()

        self.fc = nn.Sequential(
            BasicBlock(input_dim, hidden_dim),
            *[BasicBlock(hidden_dim, hidden_dim) for _ in range(hidden_layers)],
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        x = self.fc(x)
        return x

In [19]:
batch_size = 32
concat_nframes = 3

input_dim = 39 * concat_nframes
output_dim = 41

model = Classifier(
    input_dim=input_dim,
    output_dim=output_dim,
)

features = torch.randn(batch_size, input_dim)
labels = torch.randint(
    low=0,
    high=output_dim,
    size=(batch_size,),
)

logits = model(features)

print("features:", features.shape)
print("labels:", labels.shape)
print("labels dtype:", labels.dtype)
print("logits:", logits.shape)

features: torch.Size([32, 117])
labels: torch.Size([32])
labels dtype: torch.int64
logits: torch.Size([32, 41])


# Hyper-parameters

In [10]:
# data prarameters
# TODO: change the value of "concat_nframes" for medium baseline
concat_nframes = 3   # the number of frames to concat with, n must be odd (total 2k+1 = n frames)
train_ratio = 0.75   # the ratio of data used for training, the rest will be used for validation

# training parameters
seed = 1213          # random seed
batch_size = 512        # batch size
num_epoch = 10         # the number of training epoch
learning_rate = 1e-4      # learning rate
model_path = './model.ckpt'  # the path where the checkpoint will be saved

# model parameters
# TODO: change the value of "hidden_layers" or "hidden_dim" for medium baseline
input_dim = 39 * concat_nframes  # the input dim of the model, you should not change the value
hidden_layers = 2          # the number of hidden layers
hidden_dim = 64           # the hidden dim

# Dataloader

In [11]:
from torch.utils.data import DataLoader
import gc

same_seeds(seed)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'DEVICE: {device}')

# preprocess data
train_X, train_y = preprocess_data(split='train', feat_dir='./libriphone/feat', phone_path='./libriphone', concat_nframes=concat_nframes, train_ratio=train_ratio)
val_X, val_y = preprocess_data(split='val', feat_dir='./libriphone/feat', phone_path='./libriphone', concat_nframes=concat_nframes, train_ratio=train_ratio)

# get dataset
train_set = LibriDataset(train_X, train_y)
val_set = LibriDataset(val_X, val_y)

# remove raw feature to save memory
del train_X, train_y, val_X, val_y
gc.collect()

# get dataloader
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)

DEVICE: cuda


FileNotFoundError: [Errno 2] No such file or directory: './libriphone/train_labels.txt'

# Training

In [ ]:
# create model, define a loss function, and optimizer
model = Classifier(input_dim=input_dim, hidden_layers=hidden_layers, hidden_dim=hidden_dim).to(device)
criterion = nn.CrossEntropyLoss() 
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

best_acc = 0.0
for epoch in range(num_epoch):
    train_acc = 0.0
    train_loss = 0.0
    val_acc = 0.0
    val_loss = 0.0
    
    # training
    model.train() # set the model to training mode
    for i, batch in enumerate(tqdm(train_loader)):
        features, labels = batch
        features = features.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad() 
        outputs = model(features) 
        
        loss = criterion(outputs, labels)
        loss.backward() 
        optimizer.step() 
        
        _, train_pred = torch.max(outputs, 1) # get the index of the class with the highest probability
        train_acc += (train_pred.detach() == labels.detach()).sum().item()
        train_loss += loss.item()
    
    # validation
    model.eval() # set the model to evaluation mode
    with torch.no_grad():
        for i, batch in enumerate(tqdm(val_loader)):
            features, labels = batch
            features = features.to(device)
            labels = labels.to(device)
            outputs = model(features)
            
            loss = criterion(outputs, labels) 
            
            _, val_pred = torch.max(outputs, 1) 
            val_acc += (val_pred.cpu() == labels.cpu()).sum().item() # get the index of the class with the highest probability
            val_loss += loss.item()

    print(f'[{epoch+1:03d}/{num_epoch:03d}] Train Acc: {train_acc/len(train_set):3.5f} Loss: {train_loss/len(train_loader):3.5f} | Val Acc: {val_acc/len(val_set):3.5f} loss: {val_loss/len(val_loader):3.5f}')

    # if the model improves, save a checkpoint at this epoch
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), model_path)
        print(f'saving model with acc {best_acc/len(val_set):.5f}')


In [ ]:
del train_set, val_set
del train_loader, val_loader
gc.collect()

# Testing
Create a testing dataset, and load model from the saved checkpoint.

In [16]:
# load data
test_X = preprocess_data(split='test', feat_dir='./libriphone/feat', phone_path='./libriphone', concat_nframes=concat_nframes)
test_set = LibriDataset(test_X, None)
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)

FileNotFoundError: [Errno 2] No such file or directory: './libriphone/test_split.txt'

In [ ]:
# load model
model = Classifier(input_dim=input_dim, hidden_layers=hidden_layers, hidden_dim=hidden_dim).to(device)
model.load_state_dict(torch.load(model_path))

Make prediction.

In [17]:
pred = np.array([], dtype=np.int32)

model.eval()
with torch.no_grad():
    for i, batch in enumerate(tqdm(test_loader)):
        features = batch
        features = features.to(device)

        outputs = model(features)

        _, test_pred = torch.max(outputs, 1) # get the index of the class with the highest probability
        pred = np.concatenate((pred, test_pred.cpu().numpy()), axis=0)


NameError: name 'test_loader' is not defined

Write prediction to a CSV file.

After finish running this block, download the file `prediction.csv` from the files section on the left-hand side and submit it to Kaggle.

In [ ]:
with open('prediction.csv', 'w') as f:
    f.write('Id,Class\n')
    for i, y in enumerate(pred):
        f.write('{},{}\n'.format(i, y))

# 用合成数据训练

生成数据集

In [20]:
import torch
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(1213)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

num_samples = 6000
num_classes = 41
concat_nframes = 3
input_dim = 39 * concat_nframes

# 合成输入
X = torch.randn(num_samples, input_dim)

# 隐藏的“教师模型”
teacher_weight = torch.randn(input_dim, num_classes)
teacher_bias = torch.randn(num_classes)

# 根据教师模型生成标签
with torch.no_grad():
    teacher_logits = (
        X @ teacher_weight
        + teacher_bias
        + 0.1 * torch.randn(num_samples, num_classes)
    )

    y = teacher_logits.argmax(dim=1)

print("X:", X.shape)
print("y:", y.shape)
print("y dtype:", y.dtype)
print("label range:", y.min().item(), y.max().item())

X: torch.Size([6000, 117])
y: torch.Size([6000])
y dtype: torch.int64
label range: 0 40


# 划分训练集和验证集

In [21]:
train_size = int(num_samples * 0.8)

train_X = X[:train_size]
train_y = y[:train_size]

valid_X = X[train_size:]
valid_y = y[train_size:]

train_dataset = TensorDataset(train_X, train_y)
valid_dataset = TensorDataset(valid_X, valid_y)

train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True,
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=256,
    shuffle=False,
)

# Training

In [23]:
model = Classifier(
    input_dim=input_dim,
    output_dim=num_classes,
    hidden_layers=2,
    hidden_dim=256,
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
)

num_epochs = 25

for epoch in range(num_epochs):
    # -------------------------
    # Training
    # -------------------------
    model.train()

    train_loss = 0.0
    train_correct = 0

    for features, labels in train_loader:
        features = features.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(features)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * features.size(0)

        predictions = logits.argmax(dim=1)
        train_correct += (
            predictions == labels
        ).sum().item()

    train_loss /= len(train_dataset)
    train_accuracy = train_correct / len(train_dataset)

    # -------------------------
    # Validation
    # -------------------------
    model.eval()

    valid_loss = 0.0
    valid_correct = 0

    with torch.no_grad():
        for features, labels in valid_loader:
            features = features.to(device)
            labels = labels.to(device)

            logits = model(features)
            loss = criterion(logits, labels)

            valid_loss += loss.item() * features.size(0)

            predictions = logits.argmax(dim=1)
            valid_correct += (
                predictions == labels
            ).sum().item()

    valid_loss /= len(valid_dataset)
    valid_accuracy = valid_correct / len(valid_dataset)

    print(
        f"Epoch {epoch + 1:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Valid Loss: {valid_loss:.4f} | "
        f"Valid Acc: {valid_accuracy:.4f}"
    )

Epoch 01 | Train Loss: 3.7671 | Train Acc: 0.0360 | Valid Loss: 3.6332 | Valid Acc: 0.0750
Epoch 02 | Train Loss: 3.6299 | Train Acc: 0.0646 | Valid Loss: 3.5443 | Valid Acc: 0.1300
Epoch 03 | Train Loss: 3.5503 | Train Acc: 0.0831 | Valid Loss: 3.4694 | Valid Acc: 0.1608
Epoch 04 | Train Loss: 3.4644 | Train Acc: 0.1069 | Valid Loss: 3.3882 | Valid Acc: 0.1825
Epoch 05 | Train Loss: 3.3686 | Train Acc: 0.1273 | Valid Loss: 3.2920 | Valid Acc: 0.2008
Epoch 06 | Train Loss: 3.2753 | Train Acc: 0.1542 | Valid Loss: 3.1822 | Valid Acc: 0.2258
Epoch 07 | Train Loss: 3.1598 | Train Acc: 0.1800 | Valid Loss: 3.0759 | Valid Acc: 0.2550
Epoch 08 | Train Loss: 3.0353 | Train Acc: 0.2133 | Valid Loss: 2.9843 | Valid Acc: 0.2833
Epoch 09 | Train Loss: 2.9279 | Train Acc: 0.2371 | Valid Loss: 2.8910 | Valid Acc: 0.3017
Epoch 10 | Train Loss: 2.8335 | Train Acc: 0.2662 | Valid Loss: 2.7934 | Valid Acc: 0.3175
Epoch 11 | Train Loss: 2.7498 | Train Acc: 0.2796 | Valid Loss: 2.6978 | Valid Acc: 0.3350